[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/08_data_vis/08_3_One_Numeric_Column.ipynb)

# 08.3: One Numeric Column

A mean or a median reduces a whole column to one number, and notebook 08.2 showed how much that can hide. A **histogram** shows the whole column. It divides the range of the values into equal ranges called **bins**, and it draws a bar showing how many values fall in each bin. This notebook shows how to read a histogram, and it deals with the main issue in using one: what you see depends on the choices you make when you draw it.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## Reading a histogram

We use data on the Old Faithful geyser in Yellowstone National Park, where one row is one eruption. `duration` is how long the eruption lasted, in minutes.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

geyser = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_geyser.csv")
print(geyser["duration"].describe())
ax = sns.histplot(data=geyser, x="duration")
ax.set_title("Duration of Old Faithful eruptions")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of eruptions")
plt.show()

When you read a histogram, look for four things:

- **Peaks:** where do the values gather, and how many separate peaks are there? A column with one peak is called **unimodal** and one with two is called **bimodal**.
- **Shape:** is it roughly symmetric, or does it have a long tail on one side? A long tail to the right is called **right skew**.
- **Spread:** how wide a range do most of the values cover?
- **Unusual values:** are there bars sitting off by themselves?

This histogram has two clear peaks, one near 2 minutes and one near 4.5 minutes, with a gap between them. So there seem to be two kinds of eruption, short and long. The summary above could not tell you this. It gives a mean of about 3.49 minutes, which sounds like a typical eruption, but the next cell shows that very few eruptions last about that long.

In [ ]:
ax = sns.histplot(data=geyser, x="duration")
ax.set_title("The mean falls in the gap")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of eruptions")
ax.axvline(geyser["duration"].mean(), color="red", label="Mean")
ax.legend()
plt.show()
print(geyser["duration"].between(2.5, 3.5).sum())

`ax.axvline()` draws a vertical line, here at the mean. The red line sits in the gap: only 14 of the 272 eruptions lasted between 2.5 and 3.5 minutes. This is the main reason to look at a histogram before you report an average. A mean or a median does not describe a typical value when a column has two separate peaks, and the better description here is "short eruptions of about 2 minutes and long eruptions of about 4.5 minutes."

## The main issue: bin width

The **bin width** is how wide each bin is. Seaborn chooses a number of bins for you, but its choice is only a starting point, and the same column can look quite different with different bins. `bins=` sets the number of bins. Here is the same column with too few and with too many.

In [ ]:
ax = sns.histplot(data=geyser, x="duration", bins=3)
ax.set_title("3 bins")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of eruptions")
plt.show()

ax = sns.histplot(data=geyser, x="duration", bins=100)
ax.set_title("100 bins")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of eruptions")
plt.show()

With 3 bins you can still see that the middle is lower, but the chart hides where the peaks are and how nearly empty the gap is. The middle bar holds 36 eruptions, mostly because its range runs up to about 3.9 minutes and catches the start of the long eruptions. With 100 bins the opposite happens. The chart turns into a comb of thin spikes and empty spaces, and most of that detail is chance, because only 272 eruptions are spread across 100 bins. Bins that are too wide hide real features, and bins that are too narrow make noise look like features.

There is no single correct bin width. The practical approach is to try several and look for the pattern that stays when you change them. The two peaks in this column appear at nearly every reasonable setting, so you can trust them. A feature that shows up at only one setting needs a closer look. A column with more rows can support narrower bins.

`binwidth=` is another way to set this, and it is often easier to explain, since you can choose a width that means something, such as a quarter of a minute.

In [ ]:
ax = sns.histplot(data=geyser, x="duration", binwidth=0.25)
ax.set_title("Bins a quarter of a minute wide")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of eruptions")
plt.show()

The two peaks are clear, and each bar now has a plain meaning: the number of eruptions in a span of 15 seconds. Seaborn adjusts the width slightly when the range of the data does not divide evenly by it, so that the bins fit exactly. Asking for a `binwidth` of 5 on a column with a range of 53, for example, gives bins about 4.8 wide. Here the range of 3.5 divides evenly by 0.25, so the bins are exactly that wide. Every bin in a histogram must have the same width, because the height of a bar is only comparable with the others if they all cover the same range.

## When narrow bins show something real

Narrow bins are not always noise. This table has 53940 diamonds, which is plenty of rows to support narrow bins. Here are their weights, in carats, for diamonds up to 2.2 carats.

In [ ]:
diamonds = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_diamonds.csv")
smaller = diamonds[diamonds["carat"] <= 2.2]
print(len(smaller))

ax = sns.histplot(data=smaller, x="carat", binwidth=0.2)
ax.set_title("Diamond weights, bins 0.2 carats wide")
ax.set_xlabel("Weight (carats)")
ax.set_ylabel("Number of diamonds")
plt.show()

ax = sns.histplot(data=smaller, x="carat", binwidth=0.01)
ax.set_title("Diamond weights, bins 0.01 carats wide")
ax.set_xlabel("Weight (carats)")
ax.set_ylabel("Number of diamonds")
plt.show()

The wide bins show a long tail to the right, with an odd bump from 1 to 1.2 carats that they cannot explain. The narrow bins explain it: there is a row of tall spikes at or just above round weights such as 1, 1.5 and 2 carats. With this many rows, a pattern that regular is unlikely to be chance, and we can check it directly by counting the exact values.

In [ ]:
carat_counts = diamonds["carat"].value_counts()
print(carat_counts.loc[[0.99, 1.0, 1.01]])
print(carat_counts.loc[[1.49, 1.5, 1.51]])

There are only 23 diamonds at 0.99 carats, but 1558 at 1.00 and 2242 at 1.01. The same thing happens at 1.5: 11 at 1.49, then 793 at 1.50 and 807 at 1.51. Most likely, this happens because a diamond just over a round weight sells for much more than one just under it, so cutters aim to stay at or just above the round number. This is a real feature of the market, and the wide bins could only hint at it. That is why you try several widths, and why you check anything surprising against the numbers.

## Whole numbers and few distinct values

A histogram is meant for a column with many possible values. Here is one that has few. One row is one car model, and `cylinders` is the number of cylinders in its engine.

In [ ]:
cars = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_mpg.csv")
print(cars["cylinders"].value_counts().sort_index())

ax = sns.histplot(data=cars, x="cylinders")
ax.set_title("A histogram of a column with few values")
ax.set_xlabel("Cylinders")
ax.set_ylabel("Number of models")
plt.show()

ax = sns.countplot(data=cars, x="cylinders")
ax.set_title("Car models by number of cylinders")
ax.set_xlabel("Cylinders")
ax.set_ylabel("Number of models")
plt.show()

The histogram draws bars that do not line up with the values and leaves empty spaces that have no meaning, since no engine has 3.5 cylinders. The count plot shows the five values directly: 204 models have 4 cylinders, 103 have 8 and 84 have 6, with only 4 and 3 models at 3 and 5 cylinders. If a numeric column holds only a few distinct values, treat it as ordered categories and use a count plot.

## A very long tail

A long tail creates a different problem. Here are the fares paid by Titanic passengers.

In [ ]:
titanic = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv")
print(titanic["Fare"].describe())
ax = sns.histplot(data=titanic, x="Fare", bins=20)
ax.set_title("Titanic fares")
ax.set_xlabel("Fare (pounds)")
ax.set_ylabel("Number of passengers")
plt.show()

The median fare is about 14.5 and the maximum is about 512, so the axis has to stretch far enough to reach values that almost nobody paid. As a result, the first bar holds most of the passengers and tells us nothing about how fares differed among them. No choice of bin width can fix this, because the problem is the range of the axis. A simple fix is to draw the part of the data you care about in one chart and to say what you left out.

In [ ]:
under_100 = titanic[titanic["Fare"] < 100]
print(len(titanic) - len(under_100))
ax = sns.histplot(data=under_100, x="Fare", binwidth=5)
ax.set_title("Titanic fares under 100 pounds (53 higher fares left out)")
ax.set_xlabel("Fare (pounds)")
ax.set_ylabel("Number of passengers")
plt.show()

Now the differences among ordinary fares are visible, with the tallest bars below about 15 pounds and a smaller bump between 25 and 30. The title reports that 53 of the 887 passengers are left out, because a reader needs to know that. Those 53 are not errors. Leaving them out is a choice made for this one chart, and they would still matter in a question about first class fares. A more advanced fix is a **log scale**, which spaces the axis by multiples rather than by equal steps. You may meet it in later courses.

You will also see **density plots**, which draw a smooth curve in place of the bars. They depend on a smoothing setting in the same way that a histogram depends on its bin width, and they can suggest values that do not exist, so we stay with histograms in this course.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
penguins = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_penguins.csv")
titanic = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv")
cars = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_mpg.csv")

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")


def check_chart(number, name, test, hint, y_label=True):
    """Check a chart stored in a variable: it has labels, and test(chart) is True."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
        return
    chart = globals()[name]
    if not hasattr(chart, "get_title"):
        print(f"Problem {number}: not yet. Store what the seaborn function returns in {name}.")
    elif not chart.get_title() or not chart.get_xlabel() or (y_label and not chart.get_ylabel()):
        print(f"Problem {number}: not yet. Give the chart a title and label its axes.")
    else:
        try:
            ok = bool(test(chart))
        except (AttributeError, IndexError, TypeError, ValueError):
            ok = False
        print(f"Problem {number}: correct." if ok else f"Problem {number}: not yet. {hint}")

### Problem 1
Make a histogram of `Median` from `majors`, and store it in `pay_hist`. Give it a title and label both axes.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check_chart(1, "pay_hist", lambda c: sum(p.get_height() for p in c.patches) == 173 and c.get_xlim()[1] > 100000, "Use sns.histplot() with data=majors and x=\"Median\".")

### Problem 2
Look at your chart from Problem 1. How would you describe the shape of `Median`? Store your answer in a variable named `pay_shape`, as one of `"symmetric"`, `"right skew"` or `"left skew"`.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "pay_shape", "right skew", "Look for the side with the long tail.")

### Problem 3
The cell below already has code for a histogram of `flipper_length_mm` from `penguins`. Run it and look at the chart. Its 3 bins hide the shape of the column. Change the code so that it uses 20 bins, and change the title to match. Keep the chart stored in `flipper_hist`.

In [ ]:
# Problem 3
flipper_hist = sns.histplot(data=penguins, x="flipper_length_mm", bins=3)
flipper_hist.set_title("Penguin flipper length, 3 bins")
flipper_hist.set_xlabel("Flipper length (mm)")
flipper_hist.set_ylabel("Number of penguins")
plt.show()

In [ ]:
#@title Check Problem 3
check_chart(3, "flipper_hist", lambda c: len(c.patches) == 20 and "3 bins" not in c.get_title(), "Change bins=3 to bins=20, and change the title.")

### Problem 4
Look at your chart from Problem 3. How many peaks does flipper length have? Store your answer in a variable named `flipper_peaks`, as one of `"1"`, `"2"` or `"3"`.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "flipper_peaks", "2", "With 20 bins you can see a dip between two humps.")

### Problem 5
Use `.value_counts()` to find the most common exact value of `Median` in `majors`, and store it in `common_pay`.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "common_pay", majors["Median"].value_counts().index[0], ".value_counts() sorts from most common to least, so look at the first label in its index.")

### Problem 6
`cars` has a `cylinders` column with only five distinct values. Which chart shows how many models have each value? Store your answer in a variable named `cylinder_chart`, as one of `"histogram"` or `"count plot"`.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "cylinder_chart", "count plot", "A histogram would draw bins that do not line up with the values.")

### Problem 7
Create a DataFrame named `cheap_fares` with the rows of `titanic` where `Fare` is under 50, and store the number of passengers left out in a variable named `fares_left_out`. Then make a histogram of `Fare` from `cheap_fares` with a `binwidth` of 5, and store it in `fare_hist`. Give it a title that reports how many were left out, and label both axes.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check_chart(7, "fare_hist", lambda c: abs(sum(p.get_height() for p in c.patches) - (titanic["Fare"] < 50).sum()) < 0.5 and str((titanic["Fare"] >= 50).sum()) in c.get_title(), "Filter with a mask, use binwidth=5, and put the number left out in the title.")

## What's next

You can now read the shape of one column and test whether that shape depends on how you drew it. Most questions go further and ask how a column differs across groups, such as whether one species is heavier than another. Drawing one histogram per group gets hard to read quickly. Notebook 08.4 introduces the box plot, which is built for that comparison, and shows what a box plot leaves out.